# 산불 이미지 분류하기 (교과서 202~213쪽)
Ⅳ. 인공지능 프로젝트 > 3. 비정형 데이터 인공지능 프로젝트 실제 2

**시작하기 전에**
1. 메뉴 **런타임 → 런타임 유형 변경 → 하드웨어 가속기: T4 GPU → 저장** (GPU가 없으면 학습이 몇 분 더 걸려요.)
2. 맨 위 셀부터 차례로 ▶ (또는 **Shift + Enter**)를 눌러 실행해요.
3. 셀 제목의 쪽수는 교과서 쪽수예요. 「(더한 부분)」은 교과서 코드에 수업용으로 더하거나 고친 줄이에요.

# **[1] 문제 정의** (203쪽)
* 지속가능발전목표 13번 「기후 변화에 대한 영향 방지와 긴급 조치」
* 화재 예방을 위해 산불 관련 이미지 데이터를 학습하는 인공지능 모델을 제작해 본다.

# **[2] 데이터 수집** (204쪽)
① 데이터셋 내려받기

교과서는 사진을 구글 드라이브에 올린 뒤 코랩과 연결해요. 수업에서는 시간을 아끼려고 **선생님이 공유한 사진 묶음(fire.zip)을 바로 내려받아** 코랩에 풀어요. 첫 코드 셀의 `ZIP_URL`에 선생님이 알려 준 주소를 붙여 넣으세요.
* `fire/fire` 폴더: 산불 이미지 162장
* `fire/non` 폴더: 산불이 아닌 이미지 162장
* `fire/test1.jpg ~ test3.jpg`: 학습에 쓰지 않는 새 이미지

사진 출처: 교과서 제공 실습 데이터(수업용)

In [ ]:
# (수업용) 산불 이미지 데이터를 내려받아 풀기 — 10초쯤 걸려요
# 사진 출처: 교과서 제공 실습 데이터(수업용)
# 선생님이 알려 준 산불 사진 주소(구글 드라이브 공유 링크)를 따옴표 안에 붙여 넣고 실행해요
ZIP_URL = ''
assert ZIP_URL, '따옴표 안에 선생님이 알려 준 주소를 붙여 넣고 다시 실행하세요'
!gdown -q --fuzzy "$ZIP_URL" -O fire.zip
!unzip -q -o fire.zip
!ls /content/fire
# ↑ fire  non  test1.jpg  test2.jpg  test3.jpg 가 보이면 성공! 안 보이면 선생님께 알려 주세요.

② (대안) 교과서처럼 구글 드라이브와 코랩으로 연결하기 (204쪽)

내 구글 드라이브에 `fire` 폴더(안에 `fire`, `non` 폴더)를 올려 두었다면, 아래 셀의 `#`을 지우고 실행한 뒤 다음 셀의 `path`를 `'/gdrive/MyDrive/fire'`로 바꾸세요. 위에서 zip을 내려받았다면 이 셀은 건너뛰어요.

In [ ]:
# from google.colab import drive
# drive.mount('/gdrive', force_remount = True)

# **[3] 데이터 전처리** (205쪽)
① 이미지 데이터셋 확인하기

In [ ]:
import os                                     # 경로 설정과 관련된 os 라이브러리 불러오기
path = '/content/fire'                        # 이미지 데이터셋 기본 경로 설정하기 (드라이브: '/gdrive/MyDrive/fire')
train_fire_dir = path + '/fire'               # 산불 이미지 데이터 폴더 설정하기
train_fire_names = os.listdir(train_fire_dir) # 산불 이미지 데이터 목록 불러오기
print(train_fire_names[:2])                   # 2개의 산불 이미지 데이터 출력하기
print(len(train_fire_names))                  # 산불 이미지 개수 출력하기

In [ ]:
# 같은 방법으로 산불이 아닌 이미지(non 폴더) 확인하기
train_non_dir = path + '/non'
train_non_names = os.listdir(train_non_dir)
print(train_non_names[:2])
print(len(train_non_names))

② 이미지 데이터 리스트 생성하기

In [ ]:
import glob                                        # 지정한 경로의 이미지 데이터를 불러와 리스트로 반환
train_fire = glob.glob(path + '/fire' + '/*')       # 산불 이미지 데이터
train_non_fire = glob.glob(path + '/non' + '/*')    # 산불이 아닌 이미지 데이터
train_fire[:5]                                      # (더한 부분) 앞의 5개만 보기 — 교과서는 train_fire 전체

③ 이미지 데이터 색상 변환하기

`cv2.imread( )`는 사진을 **B, G, R(파랑, 초록, 빨강)** 순서로 읽어요. 화면에 제 색으로 보이게 **R, G, B** 순서로 바꾸는 함수를 만들어요.

In [ ]:
import cv2                                  # 색상 변경이나 컴퓨터 비전 관련 작업 수행
from matplotlib.pyplot import imshow
def read_img(file_path):
  img_arr = cv2.imread(file_path)
  return cv2.cvtColor(img_arr, cv2.COLOR_BGR2RGB)   # BGR 순서를 RGB 순서로 바꾼 결과 반환

# **[4] 데이터 시각화 및 분석** (206~207쪽)
① 산불 이미지 데이터 시각화하기 — `7` 대신 다른 숫자를 넣으면 다른 사진을 볼 수 있어요. (목록 순서는 컴퓨터마다 달라서 친구와 다른 사진이 나올 수 있어요.)

In [ ]:
import matplotlib.pyplot as plt
fire_img = read_img(train_fire[7])
plt.title('Fire Image')
plt.xlabel(fire_img.shape)
plt.imshow(fire_img, aspect = 'auto')
plt.show()

② 산불이 아닌 이미지 데이터 시각화하기

In [ ]:
non_fire_img = read_img(train_non_fire[7])
plt.title('Non Fire Image')
plt.xlabel(non_fire_img.shape)
plt.imshow(non_fire_img, aspect = 'auto')
plt.show()

(더 알아보기 · 선택) 여러 장을 한꺼번에 보며 **데이터에 문제가 없는지** 살펴봐요. 여러 번 실행하면 다른 사진이 나와요. 폴더에 맞지 않는 사진을 찾았나요?

In [ ]:
import random
for name, files in [('fire', train_fire), ('non', train_non_fire)]:
  plt.figure(figsize = (14, 2.4))
  for i, f in enumerate(random.sample(files, 8)):
    plt.subplot(1, 8, i + 1)
    plt.imshow(read_img(f))
    plt.title(f.split('/')[-1], fontsize = 8)
    plt.axis('off')
  plt.show()

# **[5] 알고리즘 선정** (208쪽)
산불 이미지와 산불이 아닌 이미지 같은 **비정형 데이터**를 분류하기 위해 **합성곱 신경망(CNN)** 딥러닝 모델을 활용한다.
* 컨볼루션(합성곱): 입력 데이터와 필터(커널) 간의 합성곱 연산으로 특징을 추출
* 풀링: 입력 데이터의 공간 크기를 줄여 중요한 정보를 보존

# **[6] 모델 학습** (209~210쪽)
① CNN 모델 생성하기 — 실행하면 `input_shape` 관련 경고(UserWarning)가 한 줄 나올 수 있어요. 오류가 아니니 그대로 진행해요.

In [ ]:
import tensorflow as tf
model = tf.keras.models.Sequential([
      tf.keras.layers.Conv2D(16, (3, 3), activation = 'relu', input_shape = (400, 400, 3)),
      tf.keras.layers.MaxPool2D(2, 2),
      tf.keras.layers.Conv2D(32, (3, 3), activation = 'relu'),
      tf.keras.layers.MaxPool2D(2, 2),
      tf.keras.layers.Conv2D(32, (3, 3), activation = 'relu'),
      tf.keras.layers.MaxPool2D(2, 2),
      tf.keras.layers.Flatten(),
      tf.keras.layers.Dense(128, activation = 'relu'),
      tf.keras.layers.Dense(1, activation = 'sigmoid')
])
model.summary()

② CNN 모델 컴파일하기

In [ ]:
model.compile(loss = 'binary_crossentropy', optimizer = 'adam', metrics = ['accuracy'])

③ CNN 모델 학습하기 — 정규화(rescale)와 배치 사이즈(batch_size) 설정

폴더 이름이 곧 정답(레이블)이 돼요. 폴더 이름의 알파벳 순서대로 `fire`는 0, `non`은 1이에요.

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
train_datagen = ImageDataGenerator(rescale = 1/255)   # RGB값을 0과 1 사이로 정규화
train_generator = train_datagen.flow_from_directory(
                  path, target_size = (400,400),
                  batch_size = 9, class_mode = 'binary',
                  seed = 3
)
print(train_generator.class_indices)                  # (더한 부분) 폴더 이름 → 숫자 레이블

In [ ]:
# 모델 학습 — GPU에서 1분 안팎, GPU가 없으면 4~8분 걸려요
history = model.fit(train_generator, steps_per_epoch = 36, epochs = 8, verbose = 1)

# **[7] 성능 평가 및 수정** (211~213쪽)
① 학습 후 오차와 정확도 확인하기 — 위 셀 출력에서 epoch마다 `loss`와 `accuracy`를 확인해요.

② 오차와 정확도 시각화하기

In [ ]:
import matplotlib.pyplot as plt
plt.plot(history.history['loss'], 'r--', label = 'loss')
plt.plot(history.history['accuracy'], 'b-', label = 'accuracy')
plt.xlabel('epoch')
plt.legend()
plt.show()

③ 새로운 이미지 데이터로 이미지 학습 모델 평가하기 (212~213쪽)

새 이미지는 내려받은 zip 안에 있어요(`test1.jpg` 산불, `test2.jpg` 산불 아님). 교과서처럼 `files.upload()`로 올릴 필요 없이 바로 열어요. 내 사진으로 해 보고 싶으면 맨 아래 「(선택) 내 사진 올려 보기」 셀을 써요.

In [ ]:
from PIL import Image
img1 = Image.open(path + '/test1.jpg')     # 새로운 이미지 데이터(test1.jpg) — 교과서: '/content/test1.jpg'
img1

새로운 이미지 데이터 전처리하기 — 크기를 (400, 400)으로 바꾸고, 숫자 배열로 만들고, **학습할 때처럼 255로 나누어 0~1로 정규화**한 뒤, 사진 한 장짜리 묶음으로 만들어요.

In [ ]:
from tensorflow.keras.preprocessing import image
import numpy as np
img_resize1 = img1.resize((400,400))
test_img1 = image.img_to_array(img_resize1) / 255    # (더한 부분) / 255 : 학습 데이터와 같은 0~1 범위로
test_img1 = np.expand_dims(test_img1, axis = 0)
images1 = np.vstack([test_img1])
images1.shape

④ 새로운 이미지 데이터로 테스트하기

결과는 0과 1 사이의 수예요. **0에 가까우면 산불 이미지(fire = 0), 1에 가까우면 산불이 아닌 이미지(non = 1)** 예요. 그래서 0.5를 기준으로 나눠요.

`[1.6e-14]`처럼 **e-14**가 붙어 나오면 「× 10의 −14제곱」, 곧 소수점 아래 0이 13개인 아주 작은 수예요. 거의 0이라는 뜻이에요.

In [ ]:
classes1 = model.predict(images1, batch_size = 10)
print(classes1[0])                # e-14처럼 나오면 소수점 아래 0이 13개 = 거의 0
if classes1[0][0] < 0.5:          # (고친 부분) 교과서: if classes1[0]==0:
    print('Fire Image')
else:
    print('Non Fire Image')

산불이 아닌 이미지(test2.jpg)로도 테스트해요.

In [ ]:
img2 = Image.open(path + '/test2.jpg')
img_resize2 = img2.resize((400,400))
test_img2 = image.img_to_array(img_resize2) / 255
test_img2 = np.expand_dims(test_img2, axis = 0)
images2 = np.vstack([test_img2])
classes2 = model.predict(images2, batch_size = 10)
print(classes2[0])
if classes2[0][0] < 0.5:
    print('Fire Image')
else:
    print('Non Fire Image')
img_resize2

### (선택 · 시간이 남으면) 내 사진 올려 보기
「파일 선택」을 눌러 내 사진(**JPG 또는 PNG**)을 올리면 모델이 판별해요. 아이폰 기본 사진(HEIC)은 열리지 않으니 JPG로 바꿔서 올려요. 잘못 판별하는 사진을 찾아보고, 왜 그런지 학습 데이터와 연결해 생각해 봐요. (`test3.jpg` 빨간색 건물 사진도 zip 안에 있어요: `Image.open(path + '/test3.jpg')`)

In [ ]:
from google.colab import files
uploaded = files.upload()                            # 파일 선택 → 내 사진 고르기
name = list(uploaded.keys())[0]
img = Image.open(name).convert('RGB')                # 투명 배경 사진도 RGB로
test = np.expand_dims(image.img_to_array(img.resize((400, 400))) / 255, axis = 0)
p = model.predict(test, verbose = 0)[0][0]
print(name, round(float(p), 4), 'Fire Image' if p < 0.5 else 'Non Fire Image')
img.resize((200, 200))